# LangGraph Tutorial — Wiring Multi-Agent Pipelines

This notebook walks through how we use **LangGraph** to orchestrate 5 agents into a literature review pipeline with feedback loops.

## What is LangGraph?

LangGraph is a library for building **stateful, multi-step AI applications** as directed graphs. Key concepts:

| Concept | Description |
|---------|-------------|
| **StateGraph** | A graph where nodes share and modify a typed state dictionary |
| **Node** | A function that reads state and returns a partial state update |
| **Edge** | Connects nodes — state flows from one node to the next |
| **Conditional Edge** | Routes to different nodes based on state (this is how the Critics loop) |
| **START / END** | Special nodes marking entry and exit points |

---

In [1]:
import sys
sys.path.insert(0, "..")

from langgraph.graph import StateGraph, START, END
from lit_review_agent.state import ReviewState
print("Imports OK")

Imports OK


## 1. The State Schema

All nodes in the graph share a single **typed dictionary** (`ReviewState`). Each node reads from it and returns a partial update — LangGraph merges the update into the shared state.

This is the contract between agents:

In [2]:
import inspect
from lit_review_agent.state import ReviewState, Paper

# Show the state schema
print(inspect.getsource(ReviewState))

class ReviewState(TypedDict):
    """Shared state flowing through the LangGraph graph."""

    topic: str
    search_queries: list[str]
    papers: list[Paper]
    critic_feedback: list[str]
    iteration: int
    max_iterations: int
    final_report: str | None



Key fields:
- `topic` — input from the user
- `search_queries` — queries for the current iteration (Critic can modify these)
- `papers` — accumulated corpus (grows each iteration)
- `critic_feedback` — audit trail of Critic decisions
- `iteration` / `max_iterations` — corpus loop control
- `final_report` — output markdown
- `report_critic_feedback` — audit trail of Report Critic grounding checks
- `report_iteration` / `max_report_iterations` — report revision loop control

## 2. Node Functions

Each node is a regular Python function with signature:

```python
def node_function(state: ReviewState) -> dict:
    # Read from state
    # Do work
    # Return partial update (only keys that changed)
```

LangGraph merges the returned dict into the shared state. You only return the fields you want to update.

Our 5 nodes:

In [ ]:
from lit_review_agent.graph import search_node, synthesize_node, critic_node, report_node, report_critic_node

# Each node is just a function — let's see their signatures
for name, fn in [("search", search_node), ("synthesize", synthesize_node),
                 ("critic", critic_node), ("report", report_node),
                 ("report_critic", report_critic_node)]:
    src_lines = inspect.getsource(fn).split('\n')
    # Show just the first few lines (signature + docstring)
    print(f"--- {name}_node ---")
    for line in src_lines[:3]:
        print(f"  {line}")
    print()

--- search_node ---
  def search_node(state: ReviewState) -> dict:
      """Search PubMed and Semantic Scholar for papers matching current queries."""
      queries = state["search_queries"]

--- synthesize_node ---
  def synthesize_node(state: ReviewState) -> dict:
      """Extract structured fields from all unsynthesized papers."""
      papers = state["papers"]

--- critic_node ---
  def critic_node(state: ReviewState) -> dict:
      """Review corpus coverage and decide whether to refine or approve."""
      topic = state["topic"]

--- report_node ---
  def report_node(state: ReviewState) -> dict:
      """Generate the final markdown literature review."""
      topic = state["topic"]



### What each node does:

1. **search_node** — Takes `search_queries`, calls PubMed + Semantic Scholar, deduplicates, returns `{"papers": [...]}`
2. **synthesize_node** — Takes `papers`, runs Claude extraction on unsynthesized ones, returns `{"papers": [...]}`
3. **critic_node** — Takes `papers` + `topic`, evaluates coverage, returns `{"critic_feedback": [...], "iteration": n+1}` (and optionally `{"search_queries": [...]}` if refining)
4. **report_node** — Takes `papers` + `topic`, generates markdown report, returns `{"final_report": "..."}`
5. **report_critic_node** — Takes `final_report` + `papers` + `topic`, verifies the report is grounded in source papers, returns `{"report_critic_feedback": [...], "report_iteration": n+1}`

## 3. Building the Graph

Now we wire the nodes together. The key insight: **two critics act as routers** — one for corpus coverage, one for report grounding.

```
START → search → synthesize → critic ─┐
          ↑                             │
          └── (refine: loop back) ──────┘
                                        │
       (approve) → report → report_critic ─┐
                     ↑                      │
                     └── (revise: loop) ────┘
                                            │
                             (approve) → END
```

In [ ]:
from lit_review_agent.graph import build_graph, critic_router, report_critic_router

# Step-by-step graph construction (same as build_graph() does internally)
graph = StateGraph(ReviewState)

# 1. Register nodes
graph.add_node("search", search_node)
graph.add_node("synthesize", synthesize_node)
graph.add_node("critic", critic_node)
graph.add_node("report", report_node)
graph.add_node("report_critic", report_critic_node)
print(f"Nodes: {list(graph.nodes.keys())}")

# 2. Add linear edges (A → B means B always runs after A)
graph.add_edge(START, "search")       # Entry point
graph.add_edge("search", "synthesize")
graph.add_edge("synthesize", "critic")
print("Linear edges: START→search→synthesize→critic")

# 3. Add conditional edge from critic (corpus coverage loop)
graph.add_conditional_edges(
    "critic",          # Source node
    critic_router,     # Function that returns "search" or "report"
    {                  # Map return values to target nodes
        "search": "search",
        "report": "report",
    },
)
print("Conditional edge: critic → {search, report}")

# 4. Report → Report Critic
graph.add_edge("report", "report_critic")
print("Linear edge: report→report_critic")

# 5. Conditional edge from report critic (grounding verification loop)
graph.add_conditional_edges(
    "report_critic",
    report_critic_router,
    {
        "report": "report",
        "end": END,
    },
)
print("Conditional edge: report_critic → {report, end}")

print("\nGraph built!")

## 4. The Conditional Routers

Two router functions control the feedback loops:

### Corpus Critic Router
Decides where to go after the Critic runs:
1. Have we hit `max_iterations`? → Always go to report
2. Did the Critic say "refine"? → Loop back to search with new queries
3. Otherwise → Proceed to report

### Report Critic Router
Decides where to go after the Report Critic runs:
1. Have we hit `max_report_iterations`? → Always go to END
2. Did the Report Critic say "revise"? → Loop back to report for regeneration
3. Otherwise → Proceed to END

In [ ]:
# Let's see both router logics
print("=== Corpus Critic Router ===")
print(inspect.getsource(critic_router))
print("\n=== Report Critic Router ===")
print(inspect.getsource(report_critic_router))

In [ ]:
# Test the corpus critic router with different states

# Case 1: Critic approved
state_approved = {
    "iteration": 1, "max_iterations": 3,
    "search_queries": [], "critic_feedback": ["Iteration 0: Approved"],
    "topic": "test", "papers": [], "final_report": None,
    "report_critic_feedback": [], "report_iteration": 0, "max_report_iterations": 2,
}
print(f"Approved state → {critic_router(state_approved)}")  # "report"

# Case 2: Critic wants refinement
state_refine = {
    "iteration": 1, "max_iterations": 3,
    "search_queries": ["sleep staging wearable"],
    "critic_feedback": ["Iteration 0: Refine - missing sleep studies"],
    "topic": "test", "papers": [], "final_report": None,
    "report_critic_feedback": [], "report_iteration": 0, "max_report_iterations": 2,
}
print(f"Refine state → {critic_router(state_refine)}")  # "search"

# Case 3: Max iterations reached (even if refine requested)
state_max = {
    "iteration": 3, "max_iterations": 3,
    "search_queries": ["query"],
    "critic_feedback": ["Iteration 2: Refine"],
    "topic": "test", "papers": [], "final_report": None,
    "report_critic_feedback": [], "report_iteration": 0, "max_report_iterations": 2,
}
print(f"Max iterations → {critic_router(state_max)}")  # "report"

In [ ]:
# Test the report critic router with different states
from lit_review_agent.graph import report_critic_router

# Case 1: Report approved
state_report_ok = {
    "iteration": 1, "max_iterations": 3,
    "search_queries": [], "critic_feedback": [],
    "topic": "test", "papers": [], "final_report": "# Report",
    "report_critic_feedback": ["Report iteration 0: Approved"],
    "report_iteration": 1, "max_report_iterations": 2,
}
print(f"Report approved → {report_critic_router(state_report_ok)}")  # "end"

# Case 2: Report needs revision
state_report_revise = {
    "iteration": 1, "max_iterations": 3,
    "search_queries": [], "critic_feedback": [],
    "topic": "test", "papers": [], "final_report": "# Report",
    "report_critic_feedback": ["Report iteration 0: Revise — cohort size mismatch"],
    "report_iteration": 1, "max_report_iterations": 2,
}
print(f"Report revise → {report_critic_router(state_report_revise)}")  # "report"

# Case 3: Max report iterations reached
state_report_max = {
    "iteration": 1, "max_iterations": 3,
    "search_queries": [], "critic_feedback": [],
    "topic": "test", "papers": [], "final_report": "# Report",
    "report_critic_feedback": ["Report iteration 1: Revise"],
    "report_iteration": 2, "max_report_iterations": 2,
}
print(f"Max report iterations → {report_critic_router(state_report_max)}")  # "end"

## 5. Compiling and Running

After building, we **compile** the graph into an executable app. The compiled graph:
- Validates all edges are connected
- Creates an optimized execution plan
- Provides `.invoke()` to run the full pipeline

In [ ]:
# Compile the graph
app = graph.compile()
print(f"Compiled: {type(app).__name__}")
print(f"\nThe app.invoke() method takes an initial state dict and returns the final state.")
print(f"This is what run_review() does under the hood.")

In [ ]:
from IPython.display import Image

# Visualize the compiled graph as a PNG
Image(app.get_graph().draw_mermaid_png())

## 6. The Entry Point

`run_review()` is the user-facing function that:
1. Constructs the initial state from the topic + queries
2. Compiles the graph
3. Invokes it
4. Returns the final state (with `final_report`)

In [ ]:
from lit_review_agent.graph import run_review
print(inspect.getsource(run_review))

## 7. Execution Flow Example

Here's what happens when you call `run_review("wearable AFib detection")`:

```
1. START → search_node
   - Queries: ["wearable AFib detection"]
   - Searches PubMed + S2 → finds 20 papers
   - Deduplicates → 15 unique

2. search → synthesize_node  
   - 15 papers with task=None
   - Calls Claude on each → extracts structured fields
   - Returns 15 papers with synthesis fields filled

3. synthesize → critic_node
   - Reviews corpus: "Good AFib coverage, but no sleep staging or HAR"
   - Decision: REFINE
   - Suggested queries: ["sleep staging wearable validation"]
   - Iteration: 0 → 1

4. critic → search_node (LOOP BACK)
   - Queries: ["sleep staging wearable validation"]
   - Finds 12 new papers
   - Dedup with existing → 25 total

5. search → synthesize_node
   - 10 unsynthesized papers (15 already done)
   - Synthesizes the new ones

6. synthesize → critic_node
   - Reviews: "Good coverage across AFib + sleep. Minor gap in HAR."
   - Decision: APPROVE (good enough)
   - Iteration: 1 → 2

7. critic → report_node
   - Generates markdown lit review from 25 papers
   - Sections: Intro, Methods, Comparison Table, Themes, Gaps, Conclusions, References

8. report → report_critic_node
   - Verifies every claim against source paper data
   - Checks: cited papers exist, findings match, no fabricated numbers
   - Decision: REVISE — "Report states cohort of 500 for Paper 12 but source shows 200"
   - Report iteration: 0 → 1

9. report_critic → report_node (LOOP BACK)
   - Regenerates report with corrected data

10. report → report_critic_node
    - Re-verifies: all claims now grounded
    - Decision: APPROVE
    - Report iteration: 1 → 2

11. report_critic → END
    - Final state contains final_report with verified markdown
```

## 8. Run It For Real (requires API key)

Uncomment and run to execute the full pipeline:

In [ ]:
# Uncomment to run the full pipeline:

# import logging
# logging.basicConfig(level=logging.INFO)
#
# result = run_review(
#     topic="Wearable device validation for atrial fibrillation detection",
#     initial_queries=[
#         "(wearable OR smartwatch) AND (atrial fibrillation) AND (deep learning OR machine learning) AND (validation)",
#         "(PPG) AND (AFib detection) AND (benchmark OR evaluation)",
#     ],
#     max_iterations=2,
#     max_report_iterations=2,
# )
#
# print(f"Papers: {len(result['papers'])}")
# print(f"Iterations: {result['iteration']}")
# print(f"Critic feedback: {result['critic_feedback']}")
# print(f"Report critic feedback: {result['report_critic_feedback']}")
# print(f"\n{'='*80}\n")
# print(result['final_report'])